# Excel vs Lake Q1 Intersections

Сверка пересечений уникальных `agr_id` и `ИНН` между Excel-отчетами за январь/февраль/март и `ods_alpha.scd1_agreements`.
В Lake используется month-active SA-срез.

In [ ]:
import pandas as pd
from rail_connectors.connection import connect

month_sources = [
    {
        'report_month': '2026-01-01',
        'excel_path': '/home/jovyan/documents/Equaring/Data/01_Январь_2026.xlsx',
        'header': 1,
    },
    {
        'report_month': '2026-02-01',
        'excel_path': '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx',
        'header': 0,
    },
    {
        'report_month': '2026-03-01',
        'excel_path': '/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx',
        'header': 0,
    },
]

AGR_COL_CANDIDATES = ['ID договора', 'agr_id', 'abs_agr_id']
INN_COL_CANDIDATES = ['ИНН', 'inn', 'c_inn']


In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
print('Impala connection initialized')


In [ ]:
def pick_col(columns, candidates):
    cols = list(columns)
    lower_map = {str(c).strip().lower(): c for c in cols}
    for cand in candidates:
        if cand in cols:
            return cand
        key = str(cand).strip().lower()
        if key in lower_map:
            return lower_map[key]
    return None


def norm_agr(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
        .str.strip()
        .str.replace(r'\\.0$', '', regex=True)
        .replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})
    )


def norm_inn(s: pd.Series) -> pd.Series:
    x = (
        s.astype(str)
        .str.strip()
        .str.replace(r'\\.0$', '', regex=True)
        .str.replace(r'\\D', '', regex=True)
        .replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})
    )

    def _fix_len(v):
        if pd.isna(v):
            return pd.NA
        n = len(v)
        if n == 9:
            return v.zfill(10)
        if n == 11:
            return v.zfill(12)
        if n in (10, 12):
            return v
        return pd.NA

    return x.map(_fix_len)


In [ ]:
summary_rows = []
only_lake_inn_by_month = {}
lake_rows_top5_by_month = {}

for src in month_sources:
    report_month = src['report_month']
    report_ts = pd.to_datetime(report_month)
    month_start = report_ts.strftime('%Y-%m-%d')
    month_end = (report_ts + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')

    excel_raw = pd.read_excel(src['excel_path'], header=src.get('header', 0))
    agr_col = pick_col(excel_raw.columns, AGR_COL_CANDIDATES)
    inn_col = pick_col(excel_raw.columns, INN_COL_CANDIDATES)

    if agr_col is None or inn_col is None:
        raise ValueError(
            f"[{report_month}] Не найдены колонки agr_id/ИНН. Доступные: {list(excel_raw.columns)}"
        )

    excel_df = pd.DataFrame({
        'agr_id_norm': norm_agr(excel_raw[agr_col]),
        'inn_norm': norm_inn(excel_raw[inn_col]),
    })

    with imp:
        lake_full = imp.fetch(f"""
            select
                a.*,
                c.*,
                cast(a.abs_agr_id as string) as agr_id,
                regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
            from ods_alpha.scd1_agreements a
            join ods_alpha.scd1_companies c
              on c.n_cmp = a.n_cmp_client
            where upper(trim(cast(a.acq_class as string))) = 'SA'
              and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
              and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
              and coalesce(a.ods_deleted_flg, '0') <> '1'
              and coalesce(c.ods_deleted_flg, '0') <> '1'
              and c.c_inn is not null
        """)

    if lake_full is None:
        lake_full = pd.DataFrame(columns=['agr_id', 'inn'])

    lake_full = lake_full.copy()
    lake_full['agr_id_norm'] = norm_agr(lake_full['agr_id'])
    lake_full['inn_norm'] = norm_inn(lake_full['inn'])

    set_excel_agr = set(excel_df['agr_id_norm'].dropna().drop_duplicates().tolist())
    set_excel_inn = set(excel_df['inn_norm'].dropna().drop_duplicates().tolist())
    set_lake_agr = set(lake_full['agr_id_norm'].dropna().drop_duplicates().tolist())
    set_lake_inn = set(lake_full['inn_norm'].dropna().drop_duplicates().tolist())

    agr_intersect = set_excel_agr & set_lake_agr
    agr_only_excel = set_excel_agr - set_lake_agr
    agr_only_lake = set_lake_agr - set_excel_agr

    inn_intersect = set_excel_inn & set_lake_inn
    inn_only_excel = set_excel_inn - set_lake_inn
    inn_only_lake = set_lake_inn - set_excel_inn

    only_lake_inn_by_month[report_month] = sorted(list(inn_only_lake))

    lake_rows_top5_by_month[report_month] = (
        lake_full[
            lake_full['inn_norm'].isin(inn_only_lake)
        ]
        .drop_duplicates(subset=['inn_norm'])
        .head(5)
        .copy()
    )

    summary_rows.append({
        'report_month': report_month,
        'agr_intersect_cnt': len(agr_intersect),
        'inn_intersect_cnt': len(inn_intersect),
    })

summary = pd.DataFrame(summary_rows).sort_values('report_month').reset_index(drop=True)

print('=== Пересечения Excel vs Lake по месяцам ===')
display(summary)

for month_key in summary['report_month'].tolist():
    print(f"\n===== {month_key} =====")
    print(f"ИНН только в озере: {len(only_lake_inn_by_month[month_key])}")
    print('Примеры 5 строк (полные атрибуты) для ИНН только в озере:')
    display(lake_rows_top5_by_month[month_key])
